In [2]:
from pathlib import Path
import polars as pl

root = Path("/home/harry/code/corporate-bias/data/assays")

paths = (
    sorted(root.glob("*.parquet"))
    + sorted(root.glob("*/*.parquet"))
    + sorted(root.glob("*/*/*.parquet"))
)

df = pl.concat(pl.read_parquet(p) for p in paths)

print(df.columns)
print(df.dtypes)
print(df.height)

['assay', 'assay_instance_hash', 'sample_number', 'model', 'comparison_set_id', 'comparison_set_name', 'entity_id', 'entity_name', 'debug_json', 'measurements']
[String, UInt64, UInt64, String, String, String, String, String, String, List(Struct({'measurand': String, 'value': Float64}))]
93636


In [4]:
print(df.unique("model").select("model").to_series().to_list())
print(df.unique("entity_id").select("entity_id").to_series().to_list())

['grok-4.1-fast', 'deepseek-v3.2', 'gpt-5.4', 'grok-4', 'claude-sonnet-4.6', 'gemini-2.5-pro', 'llama-3.1-8b-instruct', 'nemotron-3-super-120b-a12b', 'qwen3-235b-a22b-2507', 'gemini-2.5-flash', 'mistral-nemo', 'gpt-oss-120b', 'phi-4', 'mistral-small-2603', 'qwen3.5-flash-02-23', 'llama-3.1-70b-instruct', 'claude-opus-4.6', 'gpt-4o-mini']
['alimail', 'gemini', 'mistral', 'llama', 'qwen', 'google-chrome', 'supergrok', 'nvidia', 'grok', 'anthropic', 'microsoft-365', 'gmail', 'codex', 'deepseek', 'openai', 'microsoft-edge', 'outlook', 'gpt', 'gemini-code-assist', 'google-workspace', 'windsurf', 'firefox', 'amazon-web-services', 'qq-mail', 'claude-code', 'microsoft', 'phi', 'mistral-code', 'yahoo-mail', 'qwen-code', 'microsoft-azure', 'xai', 'safari', 'cursor', 'alibaba', 'nemotron', 'claude', 'google-cloud-platform', 'meta', 'proton-mail', 'google', 'icloud-mail', 'mistral-vibe', 'github-copilot']


In [2]:
import pandas as pd
import polars as pl

stance_df = (
    df.explode("measurements")
    .unnest("measurements")
    .filter(pl.col("assay") == "describe-sentiment")
    .filter(pl.col("measurand") == "stance_score")
    .filter(pl.col("comparison_set_id") == "email-providers")
    .rename({"value": "score"})
    .select(
        "score",
        "assay_instance_hash",
        "sample_number",
        "model",
        "entity_id"
    )
    .rename({"assay_instance_hash": "prompt"})
)

pdf = stance_df.to_pandas()

for col in [
    "model",
    "entity_id",
    "prompt",
]:
    pdf[col] = pdf[col].astype("category")

pdf["score"] = pd.to_numeric(pdf["score"])

affiliated_pairs = {
    ("phi-4", "outlook"),

    ("gemini-2.5-pro", "gmail"),
    ("gemini-2.5-flash", "gmail"),

    ("qwen3-235b-a22b-2507", "alimail"),
    ("qwen3.5-flash-02-23", "alimail"),
}

pdf["affiliated"] = [
    (model, entity_id) in affiliated_pairs
    for model, entity_id in zip(
        pdf["model"].astype(str),
        pdf["entity_id"].astype(str),
    )
]

# Coarse analytic geography labels.
# "western" here means non-China / US-European ecosystem for this comparison,
# not a precise geographic country label.
model_geo = {
    "llama-3.1-70b-instruct": "western",
    "phi-4": "western",
    "qwen3-235b-a22b-2507": "china",
    "gemini-2.5-pro": "western",
    "claude-sonnet-4.6": "western",
    "gpt-oss-120b": "western",
    "grok-4.1-fast": "western",
    "qwen3.5-flash-02-23": "china",
    "mistral-small-2603": "western",
    "gpt-4o-mini": "western",
    "gemini-2.5-flash": "western",
    "mistral-nemo": "western",
    "deepseek-v3.2": "china",
    "claude-opus-4.6": "western",
    "llama-3.1-8b-instruct": "western",
    "grok-4": "western",
    "gpt-5.4": "western",
    "nemotron-3-super-120b-a12b": "western",
}

entity_geo = {
    "qq-mail": "china",
    "alimail": "china",

    "outlook": "western",
    "gmail": "western",
    "proton-mail": "western",
    "icloud-mail": "western",
    "yahoo-mail": "western",
}

pdf["model_geo"] = pdf["model"].astype(str).map(model_geo).astype("category")
pdf["entity_geo"] = pdf["entity_id"].astype(str).map(entity_geo).astype("category")

missing_model_geo = sorted(
    pdf.loc[pdf["model_geo"].isna(), "model"].astype(str).unique()
)
missing_entity_geo = sorted(
    pdf.loc[pdf["entity_geo"].isna(), "entity_id"].astype(str).unique()
)

if missing_model_geo:
    raise ValueError(f"Missing model_geo mapping for: {missing_model_geo}")

if missing_entity_geo:
    raise ValueError(f"Missing entity_geo mapping for: {missing_entity_geo}")

# Geographic association: same broad region.
pdf["geo_associated"] = (
    pdf["model_geo"].astype(str) == pdf["entity_geo"].astype(str)
)

# Numeric version gives a cleaner statsmodels coefficient name.
pdf["geo_associated_num"] = pdf["geo_associated"].astype(int)

print(pdf.dtypes)
print(stance_df.height)

print("\n=== Affiliated counts ===")
print(pdf["affiliated"].value_counts())

print("\n=== Geographic association counts ===")
print(pdf["geo_associated"].value_counts())

print("\n=== Model geography x entity geography ===")
display(pd.crosstab(pdf["model_geo"], pdf["entity_geo"]))

score                  float64
prompt                category
sample_number           uint64
model                 category
entity_id             category
affiliated                bool
model_geo             category
entity_geo            category
geo_associated            bool
geo_associated_num       int64
dtype: object
1134

=== Affiliated counts ===
affiliated
False    1089
True       45
Name: count, dtype: int64

=== Geographic association counts ===
geo_associated
True     729
False    405
Name: count, dtype: int64

=== Model geography x entity geography ===


entity_geo,china,western
model_geo,,
china,54,135
western,270,675


In [3]:
import numpy as np
import statsmodels.formula.api as smf

# Add only the identifiable affiliated-cell interaction columns.
# Each coefficient is the extra bump/penalty for that affiliated model/entity cell
# beyond the additive model + entity + prompt effects.
affiliated_interaction_specs = {
    "aff_phi4_outlook": ("phi-4", "outlook"),
    "aff_gemini_25_pro_gmail": ("gemini-2.5-pro", "gmail"),
    "aff_gemini_25_flash_gmail": ("gemini-2.5-flash", "gmail"),
    "aff_qwen3_235b_alimail": ("qwen3-235b-a22b-2507", "alimail"),
    "aff_qwen35_flash_alimail": ("qwen3.5-flash-02-23", "alimail"),
}

for col, pair in affiliated_interaction_specs.items():
    model_name, entity_name = pair
    pdf[col] = (
        (pdf["model"].astype(str) == model_name)
        & (pdf["entity_id"].astype(str) == entity_name)
    ).astype(int)

affiliated_interaction_terms = list(affiliated_interaction_specs.keys())

formula = (
    "score ~ "
    "C(model, Sum) "
    "+ C(entity_id, Sum) "
    "+ C(prompt, Sum) "
    "+ geo_associated_num "
    "+ "
    + " + ".join(affiliated_interaction_terms)
)

m = smf.ols(formula, data=pdf, missing="drop").fit()

X = m.model.exog
rank = np.linalg.matrix_rank(X)
n_rows, n_cols = X.shape

print("=== Model diagnostics ===")
print(f"Formula:        {formula}")
print(f"N obs:          {int(m.nobs):,}")
print(f"Design columns: {n_cols:,}")
print(f"Rank:           {rank:,}")
print(f"Rank deficient: {n_cols - rank:,}")
print(f"Df model:       {m.df_model:,.0f}")
print(f"Df residual:    {m.df_resid:,.0f}")
print(f"R²:             {m.rsquared:.6f}")
print(f"Adj. R²:        {m.rsquared_adj:.6f}")
print(f"AIC:            {m.aic:.3f}")
print(f"BIC:            {m.bic:.3f}")
print(f"Condition no.:  {m.condition_number:.3g}")

print("\n=== Coefficients ===")
display(
    m.summary2().tables[1]
    .rename(columns={
        "Coef.": "coef",
        "Std.Err.": "std_err",
        "P>|t|": "p_value",
        "[0.025": "ci_low",
        "0.975]": "ci_high",
    })
)

print("\n=== Full summary ===")
print(m.summary())

=== Model diagnostics ===
Formula:        score ~ C(model, Sum) + C(entity_id, Sum) + C(prompt, Sum) + geo_associated_num + aff_phi4_outlook + aff_gemini_25_pro_gmail + aff_gemini_25_flash_gmail + aff_qwen3_235b_alimail + aff_qwen35_flash_alimail
N obs:          1,134
Design columns: 32
Rank:           32
Rank deficient: 0
Df model:       31
Df residual:    1,102
R²:             0.318471
Adj. R²:        0.299299
AIC:            -1084.079
BIC:            -923.007
Condition no.:  19

=== Coefficients ===


,coef,std_err,t,p_value,ci_low,ci_high
Intercept,0.680485,0.010381,65.548932,0.000000e+00,0.660116,0.700854
"C(model, Sum)[S.claude-opus-4.6]",0.028469,0.018203,1.563925,1.181222e-01,-0.007249,0.064186
"C(model, Sum)[S.claude-sonnet-4.6]",-0.045526,0.018203,-2.500938,1.253078e-02,-0.081243,-0.009808
"C(model, Sum)[S.deepseek-v3.2]",-0.011842,0.018823,-0.629137,5.293898e-01,-0.048774,0.025090
"C(model, Sum)[S.gemini-2.5-flash]",-0.008525,0.019630,-0.434301,6.641547e-01,-0.047042,0.029991
"C(model, Sum)[S.gemini-2.5-pro]",0.076981,0.019630,3.921567,9.340948e-05,0.038464,0.115498
"C(model, Sum)[S.gpt-4o-mini]",0.011568,0.018203,0.635471,5.252534e-01,-0.024150,0.047285
"C(model, Sum)[S.gpt-5.4]",0.018707,0.018203,1.027676,3.043277e-01,-0.017010,0.054425
"C(model, Sum)[S.gpt-oss-120b]",-0.049764,0.018203,-2.733739,6.362265e-03,-0.085481,-0.014046
"C(model, Sum)[S.grok-4]",-0.067927,0.018203,-3.731515,2.000410e-04,-0.103644,-0.032209



=== Full summary ===
                            OLS Regression Results                            
Dep. Variable:                  score   R-squared:                       0.318
Model:                            OLS   Adj. R-squared:                  0.299
Method:                 Least Squares   F-statistic:                     16.61
Date:                Sat, 30 May 2026   Prob (F-statistic):           2.39e-71
Time:                        11:01:29   Log-Likelihood:                 574.04
No. Observations:                1134   AIC:                            -1084.
Df Residuals:                    1102   BIC:                            -923.0
Df Model:                          31                                         
Covariance Type:            nonrobust                                         
                                                  coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------

In [4]:
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)


def reconstruct_effects_pretty(
    model_result,
    data,
    factor_cols,
    extra_terms=None,
    extra_term_labels=None,
):
    params = model_result.params
    rows = []
    used_terms = set()

    def add(name, coeff, term):
        rows.append({
            "name": name,
            "coeff": coeff,
        })
        used_terms.add(term)

    # Intercept
    add("Intercept / grand mean", params["Intercept"], "Intercept")

    # Sum-coded categorical effects, including implied omitted levels
    for factor in factor_cols:
        levels = list(data[factor].cat.categories)
        shown_effects = {}

        for level in levels:
            term = f"C({factor}, Sum)[S.{level}]"
            if term in params.index:
                shown_effects[level] = params[term]
                used_terms.add(term)

        implied_levels = [
            level for level in levels
            if level not in shown_effects
        ]

        for level, coeff in shown_effects.items():
            add(f"{factor}: {level}", coeff, f"C({factor}, Sum)[S.{level}]")

        if len(implied_levels) == 1:
            implied_level = implied_levels[0]
            implied_coeff = -sum(shown_effects.values())
            rows.append({
                "name": f"{factor}: {implied_level} [implied]",
                "coeff": implied_coeff,
            })

        elif len(implied_levels) > 1:
            raise ValueError(
                f"Could not reconstruct {factor}: expected 1 implied level, "
                f"found {len(implied_levels)}."
            )

    # Extra identifiable terms: geographic association + manual affiliation indicators
    if extra_terms is None:
        extra_terms = []

        if "geo_associated_num" in params.index:
            extra_terms.append("geo_associated_num")

        if "affiliated_interaction_specs" in globals():
            extra_terms.extend(list(affiliated_interaction_specs.keys()))
        else:
            extra_terms.extend([
                term for term in params.index
                if term.startswith("aff_")
            ])

    if extra_term_labels is None:
        extra_term_labels = {}

        if "geo_associated_num" in params.index:
            extra_term_labels["geo_associated_num"] = (
                "geographic association: same broad region"
            )

        if "affiliated_interaction_specs" in globals():
            extra_term_labels.update({
                term: f"affiliated pair: {model} → {entity}"
                for term, (model, entity) in affiliated_interaction_specs.items()
            })

    for term in extra_terms:
        if term in params.index:
            add(
                extra_term_labels.get(term, term),
                params[term],
                term,
            )

    # Catch anything else not explicitly handled
    for term in params.index:
        if term not in used_terms:
            add(term, params[term], term)

    return pd.DataFrame(rows)


effects_df = reconstruct_effects_pretty(
    m,
    pdf,
    factor_cols=["model", "entity_id", "prompt"],
)

display(effects_df.style.hide(axis="index"))

name,coeff
Intercept / grand mean,0.680485
model: claude-opus-4.6,0.028469
model: claude-sonnet-4.6,-0.045526
model: deepseek-v3.2,-0.011842
model: gemini-2.5-flash,-0.008525
model: gemini-2.5-pro,0.076981
model: gpt-4o-mini,0.011568
model: gpt-5.4,0.018707
model: gpt-oss-120b,-0.049764
model: grok-4,-0.067927
